# Planilla Manuscrita → Excel
### Seguimiento de Siembra a Cosecha

Este notebook convierte un PDF de planillas manuscritas escaneadas a un archivo Excel estructurado, usando Llama 4 Vision (via Groq) para reconocer la escritura a mano.

---

## Antes de empezar: obtené tu API key gratuita de Groq (sin tarjeta)

1. Creá una cuenta en **[console.groq.com](https://console.groq.com)** con tu Gmail
2. Andá a **API Keys → Create API key**
3. Copiá la key (empieza con `gsk_...`)
4. Pegala en la celda de configuración más abajo

> Groq es completamente gratuito, sin tarjeta de crédito. Límite: 1000 requests/día — más que suficiente para las 44 páginas.

---

## Cómo usar este notebook

| Paso | Qué hacer |
|---|---|
| 1 | Pegá tu API key de Groq en la **Celda 1** |
| 2 | Hacé clic en `Runtime → Run all` para ejecutar todo |
| 3 | Cuando aparezca el botón de subir archivo, seleccioná tu PDF |
| 4 | Esperá ~5 minutos (44 páginas) |
| 5 | El Excel se descarga automáticamente al terminar |

---

In [ ]:
# @title Celda 1 — Configuración: pegá tu API key de Groq acá
API_KEY = ""  # @param {type:"string"}

if not API_KEY:
    raise ValueError("Pegá tu API key de Groq. Obtenerla gratis en: https://console.groq.com")

In [ ]:
# Celda 2 — Instalación de dependencias
!pip install -q groq PyMuPDF openpyxl pandas
print("Dependencias instaladas correctamente.")

In [ ]:
# Celda 3 — Subir el PDF
from google.colab import files

print("Seleccioná el archivo PDF con las planillas escaneadas...")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No se subió ningún archivo.")

pdf_path = list(uploaded.keys())[0]
print(f"Archivo cargado: {pdf_path}")

In [ ]:
# Celda 4 — Convertir páginas del PDF a imágenes
import fitz  # PyMuPDF
import os

def pdf_to_images(pdf_path, output_dir="pages", dpi=100):
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(pdf_path)
    image_paths = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img_path = f"{output_dir}/page_{i+1:03d}.jpg"
        pix.save(img_path, jpg_quality=85)
        image_paths.append(img_path)
    doc.close()
    return image_paths

image_paths = pdf_to_images(pdf_path)
print(f"Convertidas {len(image_paths)} páginas a imágenes.")
print(f"Tiempo estimado: ~{len(image_paths) * 4 // 60} min {len(image_paths) * 4 % 60} seg")

In [ ]:
# Celda 5 — Extracción de datos con Llama 4 Vision (Groq)
from groq import Groq
import base64
import json
import re
import time

client = Groq(api_key=API_KEY)

PROMPT = """Esta es una planilla manuscrita de 'SEGUIMIENTO DE SIEMBRA A COSECHA'.
Extraé todos los datos de la tabla y devolvé ÚNICAMENTE un JSON válido, sin texto adicional.

Columnas: VARIEDAD, NRO_ESP, FECHA_MOJ, NRO_CAJ, PORC_FALL, FECHA_CH, Z_CH,
FECHA_TQE, NRO_T, NRO_C, LADO, TRASP, DIAS_7, DIAS_14, DIAS_21, DIAS_28, COSECHA, RESTO

Estructura JSON requerida:
{
  "semana": <número o null>,
  "año": <número o null>,
  "filas": [
    {
      "VARIEDAD": "", "NRO_ESP": "", "FECHA_MOJ": "", "NRO_CAJ": "",
      "PORC_FALL": "", "FECHA_CH": "", "Z_CH": "", "FECHA_TQE": "",
      "NRO_T": "", "NRO_C": "", "LADO": "", "TRASP": "",
      "DIAS_7": "", "DIAS_14": "", "DIAS_21": "", "DIAS_28": "",
      "COSECHA": "", "RESTO": ""
    }
  ]
}

REGLAS:
- Cada fila del JSON = una fila de datos de la tabla
- Si una variedad tiene múltiples sub-filas (distintas FECHA_TQE), repetí VARIEDAD, NRO_ESP,
  FECHA_MOJ, NRO_CAJ, PORC_FALL, FECHA_CH, Z_CH en cada sub-fila
- Usá "" para celdas vacías o ilegibles
- Devolvé SOLO el JSON, sin markdown, sin explicaciones"""


def extract_page_data(image_path):
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{img_b64}"
                            }
                        },
                        {"type": "text", "text": PROMPT}
                    ]
                }],
                max_tokens=8192
            )
            text = response.choices[0].message.content.strip()
            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'^```\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text)
        except Exception as e:
            print(f"  Intento {attempt+1} fallido: {e}")
            time.sleep(5)
    return None


all_rows = []
errors = []

for i, img_path in enumerate(image_paths):
    print(f"Página {i+1:02d}/{len(image_paths)}", end=" ... ")
    data = extract_page_data(img_path)
    if data:
        filas = data.get("filas", [])
        for row in filas:
            row["SEMANA"] = data.get("semana", "")
            row["AÑO"] = data.get("año", "")
            row["PAGINA"] = i + 1
        all_rows.extend(filas)
        print(f"OK — {len(filas)} filas")
    else:
        errors.append(i + 1)
        print("ERROR")
    time.sleep(3)  # Groq free tier: 30 req/min

print(f"\nTotal filas extraídas: {len(all_rows)}")
if errors:
    print(f"Páginas con error (revisar manualmente): {errors}")
else:
    print("Todas las páginas procesadas correctamente.")

In [ ]:
# Celda 6 — Guardar en Excel y descargar
import pandas as pd
from google.colab import files

df = pd.DataFrame(all_rows)

column_order = [
    "SEMANA", "AÑO", "PAGINA",
    "VARIEDAD", "NRO_ESP", "FECHA_MOJ", "NRO_CAJ", "PORC_FALL",
    "FECHA_CH", "Z_CH", "FECHA_TQE", "NRO_T", "NRO_C",
    "LADO", "TRASP", "DIAS_7", "DIAS_14", "DIAS_21", "DIAS_28",
    "COSECHA", "RESTO"
]
df = df.reindex(columns=column_order)

output_file = "seguimiento_siembra_cosecha.xlsx"
df.to_excel(output_file, index=False)

print(f"Excel generado: {output_file}")
print(f"Filas totales: {len(df)} | Columnas: {len(df.columns)}")
print("\nVista previa de las primeras 10 filas:")
display(df.head(10))

print("\nDescargando archivo...")
files.download(output_file)
print("Listo.")